## Web Datamining LAB 1 — Justine SCHUTTERLE & POUPAT Emilie DIA 5

**Domaine : Prix Nobel**

Nous avons changé de sujet des Jeux Olympiques vers les **Prix Nobel** pour les raisons suivantes :
- Le sujet JO était trop populaire sur Wikidata → timeouts SPARQL systématiques
- Les Prix Nobel offrent ~1 000 lauréats bien documentés → expansion facile vers 50 000–200 000 triples
- Pages Wikipedia en français abondantes et structurées
- Relations riches : lauréat, institution, domaine, nationalité, co-lauréats, année

### Phase 1: Web Crawling & Cleaning

#### 1.1 Source Selection

Nous travaillons sur le domaine des **Prix Nobel**. Voici les 10 URLs seed sélectionnées, toutes en français, couvrant les différentes catégories de prix :

- https://fr.wikipedia.org/wiki/Prix_Nobel
- https://fr.wikipedia.org/wiki/Prix_Nobel_de_physique
- https://fr.wikipedia.org/wiki/Prix_Nobel_de_chimie
- https://fr.wikipedia.org/wiki/Prix_Nobel_de_physiologie_ou_m%C3%A9decine
- https://fr.wikipedia.org/wiki/Prix_Nobel_de_la_paix
- https://fr.wikipedia.org/wiki/Prix_Nobel_de_litt%C3%A9rature
- https://fr.wikipedia.org/wiki/Prix_Nobel_de_sciences_%C3%A9conomiques
- https://fr.wikipedia.org/wiki/Alfred_Nobel
- https://fr.wikipedia.org/wiki/Liste_des_laur%C3%A9ats_du_prix_Nobel
- https://fr.wikipedia.org/wiki/Acad%C3%A9mie_royale_des_sciences_de_Su%C3%A8de

**Pourquoi ce choix ?**
- Données stables (pas d'événements en cours) → Wikidata SPARQL ne timeout pas
- Pages Wikipedia riches en texte, bien structurées, facilement extractibles
- ~1 000 lauréats dans Wikidata → suffisant pour l'expansion KGE
- Relations variées et sémantiquement riches

In [1]:
# Install dependencies
import sys
!{sys.executable} -m pip install -q trafilatura spacy SPARQLWrapper rdflib pandas requests owlready2 pykeen torch scikit-learn matplotlib
!{sys.executable} -m spacy download fr_core_news_lg --quiet
print("Dependencies installed.")



[notice] A new release of pip is available: 24.0 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


✔ Download and installation successful
You can now load the package via spacy.load('fr_core_news_lg')
Dependencies installed.



[notice] A new release of pip is available: 24.0 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


#### 1.2 The Extraction Pipeline

In [2]:
# List of URLs to be crawled — Prix Nobel domain
seed_urls = [
    "https://fr.wikipedia.org/wiki/Prix_Nobel",
    "https://fr.wikipedia.org/wiki/Prix_Nobel_de_physique",
    "https://fr.wikipedia.org/wiki/Prix_Nobel_de_chimie",
    "https://fr.wikipedia.org/wiki/Prix_Nobel_de_physiologie_ou_m%C3%A9decine",
    "https://fr.wikipedia.org/wiki/Prix_Nobel_de_la_paix",
    "https://fr.wikipedia.org/wiki/Prix_Nobel_de_litt%C3%A9rature",
    "https://fr.wikipedia.org/wiki/Prix_Nobel_de_sciences_%C3%A9conomiques",
    "https://fr.wikipedia.org/wiki/Alfred_Nobel",
    "https://fr.wikipedia.org/wiki/Liste_des_laur%C3%A9ats_du_prix_Nobel",
    "https://fr.wikipedia.org/wiki/Acad%C3%A9mie_royale_des_sciences_de_Su%C3%A8de",
]
print(f"Seed URLs configured: {len(seed_urls)} pages")


Seed URLs configured: 10 pages


In [3]:
# Libraries
import json
from pathlib import Path
import trafilatura

# Creating result_file to store the results of the crawling process
result_file = Path("crawler_output.jsonl")

def extract_main_text(url: str):
    """
    Fetch a URL and extract its main textual content using trafilatura.
    Returns None if extraction fails or content is too short.
    trafilatura is specifically designed to isolate the 'main content'
    of a web page, removing navigation, ads, and boilerplate.
    """
    downloaded = trafilatura.fetch_url(url)
    if not downloaded:
        return None
    text = trafilatura.extract(
        downloaded,
        include_comments=False,   # Exclude comment sections
        include_tables=False,      # Exclude HTML tables (often noisy)
        favor_precision=True       # Prefer clean text over quantity
    )
    return text

def is_useful(text: str, min_words: int = 500) -> bool:
    """
    Filter pages by minimum word count.
    Pages with fewer than 500 words are too short for meaningful NER/relation extraction.
    """
    if not text:
        return False
    return len(text.split()) >= min_words

results = []

for url in seed_urls:
    print(f"Processing: {url}")
    text = extract_main_text(url)

    if not text:
        print("  -> extraction failed (dynamic content or network error)")
        continue

    word_count = len(text.split())

    if not is_useful(text):
        print(f"  -> skipped ({word_count} words < 500 minimum)")
        continue

    record = {"url": url, "word_count": word_count, "text": text}
    results.append(record)
    print(f"  -> kept ({word_count} words)")

with result_file.open("w", encoding="utf-8") as file:
    for record in results:
        file.write(json.dumps(record, ensure_ascii=False) + "\n")

print(f"\nSaved {len(results)} pages to {result_file}")


Processing: https://fr.wikipedia.org/wiki/Prix_Nobel
  -> kept (4372 words)
Processing: https://fr.wikipedia.org/wiki/Prix_Nobel_de_physique
  -> kept (1900 words)
Processing: https://fr.wikipedia.org/wiki/Prix_Nobel_de_chimie
  -> kept (874 words)
Processing: https://fr.wikipedia.org/wiki/Prix_Nobel_de_physiologie_ou_m%C3%A9decine
  -> kept (3491 words)
Processing: https://fr.wikipedia.org/wiki/Prix_Nobel_de_la_paix
  -> kept (4928 words)
Processing: https://fr.wikipedia.org/wiki/Prix_Nobel_de_litt%C3%A9rature
  -> kept (6124 words)
Processing: https://fr.wikipedia.org/wiki/Prix_Nobel_de_sciences_%C3%A9conomiques
  -> extraction failed (dynamic content or network error)
Processing: https://fr.wikipedia.org/wiki/Alfred_Nobel
  -> kept (2405 words)
Processing: https://fr.wikipedia.org/wiki/Liste_des_laur%C3%A9ats_du_prix_Nobel
  -> kept (504 words)
Processing: https://fr.wikipedia.org/wiki/Acad%C3%A9mie_royale_des_sciences_de_Su%C3%A8de
  -> kept (763 words)

Saved 9 pages to crawler_ou

In [4]:
# Display the first few lines of the result file to verify the output
with result_file.open("r", encoding="utf-8") as file:
    for i, line in enumerate(file):
        record = json.loads(line)
        print(f"URL: {record['url']}")
        print(f"Words: {record['word_count']}")
        print(f"Text preview: {record['text'][:200]}...")
        print()
        if i >= 2:
            break


URL: https://fr.wikipedia.org/wiki/Prix_Nobel
Words: 4372
Text preview: Le prix Nobel (Nobelpriset) est une récompense de portée internationale. Remis pour la première fois en 1901, les prix sont décernés chaque année à des personnes « ayant apporté le plus grand bénéfice...

URL: https://fr.wikipedia.org/wiki/Prix_Nobel_de_physique
Words: 1900
Text preview: Le prix Nobel de physique est une récompense attribuée par la fondation Nobel, selon les dernières volontés du testament du chimiste Alfred Nobel. Il distingue des figures scientifiques éminentes ayan...

URL: https://fr.wikipedia.org/wiki/Prix_Nobel_de_chimie
Words: 874
Text preview: Le prix Nobel de chimie est une récompense décernée une fois par an, depuis 1901, par l'Académie royale des sciences de Suède à un scientifique dont l'œuvre et les travaux ont rendu de grands services...



**Note technique :** Certaines pages peuvent échouer si Wikipedia renvoie un captcha ou si la page contient trop de JavaScript. Dans ce cas, `trafilatura.fetch_url()` retourne `None` et on passe à l'URL suivante.

### Phase 2: Information Extraction

#### 2.1 Named Entity Recognition (NER)

Nous utilisons **fr_core_news_lg** (modèle français de spaCy) car toutes nos pages sources sont en français. Ce modèle reconnaît les labels : PER (personnes), ORG (organisations), LOC (lieux), MISC (entités diverses).

In [5]:
import csv
import json
import spacy
from pathlib import Path

# Load the French NLP model
# We use fr_core_news_lg rather than en_core_web_trf because all sources are in French.
# fr_core_news_lg includes: transformer, tagger, parser, ner, attribute_ruler, lemmatizer
nlp = spacy.load("fr_core_news_lg")

LABELS_TO_KEEP = {"PER", "ORG", "LOC", "MISC"}  # Person, Organization, Location, Miscellaneous

input_file  = Path("crawler_output.jsonl")
output_file = Path("extracted_knowledge.csv")

entities_found = []

with input_file.open("r", encoding="utf-8") as f:
    for line in f:
        record = json.loads(line)
        url    = record["url"]
        text   = record["text"]

        # Process in chunks of 100 000 chars to avoid spaCy memory issues on long pages
        for i in range(0, len(text), 100_000):
            chunk = text[i:i+100_000]
            doc   = nlp(chunk)

            for ent in doc.ents:
                if ent.label_ not in LABELS_TO_KEEP:
                    continue
                # Filter very short or fully lowercase entities (likely not proper nouns)
                if len(ent.text.strip()) < 2 or ent.text.islower():
                    continue
                entities_found.append({
                    "entity":     ent.text.strip(),
                    "label":      ent.label_,
                    "source_url": url,
                })

print(f"{len(entities_found)} entities extracted")

with output_file.open("w", encoding="utf-8", newline="") as f:
    writer = csv.DictWriter(f, fieldnames=["entity", "label", "source_url"])
    writer.writeheader()
    writer.writerows(entities_found)

print(f"Saved to {output_file}")


2378 entities extracted
Saved to extracted_knowledge.csv


#### 2.2 Introduction to Relations

Nous extrayons des triplets Sujet–Verbe–Objet (SVO) en cherchant des verbes dont le sujet et l'objet dépendanciel sont tous deux des entités nommées. Le domaine Nobel se prête bien à cette approche : *"Marie Curie a reçu le Prix Nobel de physique"*.

In [6]:
import json
from pathlib import Path
import csv
import spacy

nlp = spacy.load("fr_core_news_lg")

input_file             = Path("crawler_output.jsonl")
output_file_relations  = Path("extracted_relations.csv")

relations_found = []

with input_file.open("r", encoding="utf-8") as f:
    for line in f:
        record = json.loads(line)
        url    = record["url"]
        text   = record["text"]
        doc    = nlp(text[:50_000])  # Limit to first 50k chars for speed

        for sent in doc.sents:
            ents_in_sent = [
                ent for ent in sent.ents
                if ent.label_ in {"PER", "ORG", "LOC", "MISC"}
            ]
            if len(ents_in_sent) < 2:
                continue

            for token in sent:
                if token.pos_ != "VERB":
                    continue

                subject = None
                for child in token.children:
                    if child.dep_ == "nsubj":
                        subject = child
                        break

                obj = None
                for child in token.children:
                    if child.dep_ in {"dobj", "obj"}:
                        obj = child
                        break

                if not subject or not obj:
                    continue

                subject_ent = None
                obj_ent     = None
                for ent in ents_in_sent:
                    if subject.i >= ent.start and subject.i < ent.end:
                        subject_ent = ent
                    if obj.i >= ent.start and obj.i < ent.end:
                        obj_ent = ent

                if not subject_ent or not obj_ent:
                    continue

                relations_found.append({
                    "subject":       subject_ent.text.strip(),
                    "subject_label": subject_ent.label_,
                    "relation":      token.lemma_,
                    "object":        obj_ent.text.strip(),
                    "object_label":  obj_ent.label_,
                    "source_url":    url,
                })

print(f"{len(relations_found)} relations found")

with output_file_relations.open("w", encoding="utf-8", newline="") as f:
    writer = csv.DictWriter(f, fieldnames=["subject","subject_label","relation","object","object_label","source_url"])
    writer.writeheader()
    writer.writerows(relations_found)

print(f"Saved to {output_file_relations}")


15 relations found
Saved to extracted_relations.csv


**Note :** Le faible nombre de relations SVO extraites est normal : le pipeline est strict (verbe + nsubj + dobj, tous deux des entités). Cette rigueur garantit la qualité des triples plutôt que la quantité. L'expansion SPARQL (Phase 6) fournira le volume nécessaire pour le KGE.

### Phase 3: Knowledge Base Construction (Step 1)

Conversion des entités et relations extraites en un graphe RDF formel. Chaque connaissance est représentée comme un triplet : **Sujet → Prédicat → Objet**.

In [7]:
import re
import pandas as pd
from rdflib import Graph, Namespace, URIRef, Literal, RDF, RDFS, OWL
from rdflib.namespace import XSD

# --- Namespace definition ---
# Private namespace for all Nobel-domain entities and predicates
NOBEL  = Namespace("http://nobel-kb.org/entity/")
PROP   = Namespace("http://nobel-kb.org/property/")
SCHEMA = Namespace("http://schema.org/")

g = Graph()
g.bind("nobel",  NOBEL)
g.bind("prop",   PROP)
g.bind("schema", SCHEMA)
g.bind("owl",    OWL)
g.bind("rdfs",   RDFS)

print("Namespaces defined.")


Namespaces defined.


In [8]:
def to_uri(text: str) -> str:
    """
    Convert a raw text string into a valid URI fragment (CamelCase).
    Example: 'Prix Nobel de physique' -> 'PrixNobelDePhysique'
    Handles accented characters by simple capitalization.
    """
    text  = text.strip()
    parts = re.split(r'[\s\-\'\"]+', text)
    camel = "".join(word.capitalize() for word in parts if word)
    camel = re.sub(r'[^a-zA-Z0-9_]', '', camel)
    return camel if camel else "Unknown"

# NER label to schema.org class mapping
LABEL_TO_CLASS = {
    "PER":  SCHEMA.Person,
    "ORG":  SCHEMA.Organization,
    "LOC":  SCHEMA.Place,
    "MISC": SCHEMA.Thing,
}

# French verb lemma to predicate URI mapping (Nobel domain)
RELATION_TO_PRED = {
    "recevoir":   PROP.receives,
    "obtenir":    PROP.obtains,
    "décerner":   PROP.awards,
    "attribuer":  PROP.attributes,
    "créer":      PROP.creates,
    "fonder":     PROP.founds,
    "travailler": PROP.worksAt,
    "enseigner":  PROP.teachesAt,
    "diriger":    PROP.leads,
    "publier":    PROP.publishes,
    "découvrir":  PROP.discovers,
    "naître":     PROP.bornIn,
    "mourir":     PROP.diedIn,
    "étudier":    PROP.studiedAt,
}

print("URI helpers and mappings ready.")


URI helpers and mappings ready.


In [9]:
# --- Step 1a: Add SVO relation triples ---
relations_df = pd.read_csv("extracted_relations.csv")

svo_count = 0
for _, row in relations_df.iterrows():
    subj_uri = NOBEL[to_uri(row["subject"])]
    obj_uri  = NOBEL[to_uri(row["object"])]
    pred_uri = RELATION_TO_PRED.get(row["relation"], PROP[to_uri(row["relation"])])

    g.add((subj_uri, pred_uri, obj_uri))

    subj_class = LABEL_TO_CLASS.get(row["subject_label"], SCHEMA.Thing)
    obj_class  = LABEL_TO_CLASS.get(row["object_label"],  SCHEMA.Thing)
    g.add((subj_uri, RDF.type, subj_class))
    g.add((obj_uri,  RDF.type, obj_class))

    g.add((subj_uri, RDFS.label, Literal(row["subject"], lang="fr")))
    g.add((obj_uri,  RDFS.label, Literal(row["object"],  lang="fr")))
    g.add((subj_uri, PROP.extractedFrom, URIRef(row["source_url"])))

    svo_count += 1

print(f"SVO triples processed: {svo_count} relations -> {len(g)} RDF triples so far")


SVO triples processed: 15 relations -> 78 RDF triples so far


In [10]:
# --- Step 1b: Add NER entity triples ---
entities_df    = pd.read_csv("extracted_knowledge.csv")
unique_entities = entities_df.drop_duplicates(subset=["entity", "label"])

ner_count = 0
for _, row in unique_entities.iterrows():
    entity_uri   = NOBEL[to_uri(row["entity"])]
    entity_class = LABEL_TO_CLASS.get(row["label"], SCHEMA.Thing)

    g.add((entity_uri, RDF.type, entity_class))
    g.add((entity_uri, RDFS.label, Literal(row["entity"], lang="fr")))
    g.add((entity_uri, PROP.extractedFrom, URIRef(row["source_url"])))
    ner_count += 1

print(f"Unique NER entities processed: {ner_count}")
print(f"Total RDF triples in the graph: {len(g)}")


Unique NER entities processed: 1477
Total RDF triples in the graph: 4403


In [11]:
# --- Step 1c: KB Statistics ---
all_subjects   = set(g.subjects())
all_objects    = set(o for o in g.objects() if isinstance(o, URIRef))
all_entities   = all_subjects | all_objects
all_predicates = set(g.predicates())

print("=== Knowledge Base Statistics ===")
print(f"  Total triples   : {len(g)}")
print(f"  Unique entities : {len(all_entities)}")
print(f"  Unique relations: {len(all_predicates)}")
print()
print("Requirements check:")
print(f"  >= 100 triples  : {'OK' if len(g) >= 100 else 'FAIL'}  ({len(g)} triples)")
print(f"  >= 50 entities  : {'OK' if len(all_entities) >= 50 else 'FAIL'}  ({len(all_entities)} entities)")


=== Knowledge Base Statistics ===
  Total triples   : 4403
  Unique entities : 1455
  Unique relations: 12

Requirements check:
  >= 100 triples  : OK  (4403 triples)
  >= 50 entities  : OK  (1455 entities)


In [12]:
# --- Step 1d: Serialize the Knowledge Base ---
from pathlib import Path

output_nt  = Path("nobel_kb.nt")
output_ttl = Path("nobel_kb.ttl")

g.serialize(destination=str(output_nt),  format="nt")
g.serialize(destination=str(output_ttl), format="turtle")

print(f"Knowledge Base saved:")
print(f"  {output_nt}  (N-Triples format — for KGE)")
print(f"  {output_ttl} (Turtle format — human-readable)")


c:\Users\user\AppData\Local\Programs\Python\Python311\Lib\site-packages\rdflib\plugins\serializers\nt.py:39: UserWarning: NTSerializer always uses UTF-8 encoding. Given encoding was: None
  warnings.warn(


Knowledge Base saved:
  nobel_kb.nt  (N-Triples format — for KGE)
  nobel_kb.ttl (Turtle format — human-readable)


In [13]:
# --- Step 1e: Preview sample triples ---
print("Sample of generated triples (Turtle format):")
print("-" * 60)
for i, (s, p, o) in enumerate(g):
    s_short = g.namespace_manager.qname(s) if isinstance(s, URIRef) else str(s)
    p_short = g.namespace_manager.qname(p) if isinstance(p, URIRef) else str(p)
    o_short = g.namespace_manager.qname(o) if isinstance(o, URIRef) else str(o)
    print(f"  {s_short:<40} {p_short:<25} {o_short}")
    if i >= 14:
        break


Sample of generated triples (Turtle format):
------------------------------------------------------------
  nobel:Sude9                              prop:extractedFrom        ns1:Alfred_Nobel
  nobel:DrAlfredNobel                      rdfs:label                Dr Alfred Nobel
  nobel:TheNobelPrizeInChemistry2018       rdf:type                  schema1:Thing
  nobel:MetroLondon                        rdfs:label                Metro London
  nobel:AllNobelPrizesInLiterature         rdfs:label                All Nobel Prizes in Literature
  nobel:ManifestationsDeLaPlaceTianAnmen   rdfs:label                manifestations de la place Tian'anmen
  nobel:CamiloJosCela                      prop:extractedFrom        ns1:Prix_Nobel_de_litt%C3%A9rature
  nobel:RobertKoch                         rdfs:label                Robert Koch
  nobel:Chinois                            rdf:type                  schema1:Place
  nobel:HenrikNicander                     prop:extractedFrom        ns1:Acad%C3%A9

**Summary of Phase 3 — Step 1:**

Nous avons construit une KB RDF privée sur les Prix Nobel en utilisant deux sources complémentaires :
- Les relations SVO extraites par parsage dépendanciel, converties en triples sémantiques avec URIs propres
- Les entités nommées uniques issues de la NER (dédupliquées), typées selon schema.org

Le graphe initial sert de **noyau sémantique** qui sera aligné avec Wikidata puis massivement étendu en Phase 6.


### Phase 4: Entity Linking with Open Knowledge Bases (Step 2)

Pour chaque entité de notre KB privée, nous cherchons sa contrepartie dans **Wikidata** via l'API `wbsearchentities`.

**Stratégie d'amélioration par rapport au sujet JO :**  
Le problème précédent était que les entités courtes comme "CIO" ou "JO" retournaient 0 résultats car trop ambiguës.  
Pour les Prix Nobel, on enrichit la requête avec le contexte du domaine : on cherche en français (`lang="fr"`) et on valide par la description retournée.


In [14]:
import time
import requests
from rdflib import Graph, Namespace, URIRef, Literal, RDF, RDFS, OWL
import pandas as pd
from pathlib import Path

WIKIDATA_API = "https://www.wikidata.org/w/api.php"
WD   = Namespace("http://www.wikidata.org/entity/")
NOBEL = Namespace("http://nobel-kb.org/entity/")
PROP  = Namespace("http://nobel-kb.org/property/")

def search_wikidata(label: str, lang: str = "fr", domain_keywords: list = None) -> dict | None:
    """
    Search Wikidata for an entity by label using wbsearchentities API.
    
    Improvement over the Olympics version:
    - Uses domain_keywords to boost confidence when the description
      mentions Nobel-related terms (e.g., 'prix Nobel', 'lauréat', 'chimie')
    - This addresses the 'all NOT FOUND' bug from the Olympics version
      where entities were too ambiguous without domain context.
    
    Returns: dict with qid, uri, description, confidence — or None if no match.
    """
    params = {
        "action":   "wbsearchentities",
        "search":   label,
        "language": lang,
        "format":   "json",
        "limit":    5,  # Fetch top 5 for better chance of finding Nobel context
    }
    try:
        response = requests.get(WIKIDATA_API, params=params, timeout=10)
        data     = response.json()
    except Exception:
        return None

    results = data.get("search", [])
    if not results:
        return None

    domain_keywords = domain_keywords or ["nobel", "prix", "physique", "chimie",
                                           "médecine", "paix", "littérature",
                                           "université", "académie", "lauréat",
                                           "scientifique", "chercheur"]

    label_lower = label.lower()

    # Score each candidate: prefer domain context + exact label match
    best_result    = None
    best_confidence = 0.0

    for candidate in results:
        cand_label = candidate.get("label", "").lower()
        cand_desc  = candidate.get("description", "").lower()

        # Base confidence
        if label_lower == cand_label:
            confidence = 1.0
        elif label_lower in cand_label or cand_label in label_lower:
            confidence = 0.8
        else:
            confidence = 0.5

        # Boost if description contains Nobel-domain keywords
        # This fixes the 'all NOT FOUND' issue from Olympics version
        domain_boost = any(kw in cand_desc for kw in domain_keywords)
        if domain_boost:
            confidence = min(confidence + 0.15, 1.0)

        if confidence > best_confidence:
            best_confidence = confidence
            best_result     = candidate

    if best_result is None:
        return None

    return {
        "qid":           best_result["id"],
        "uri":           WD[best_result["id"]],
        "matched_label": best_result.get("label", ""),
        "description":   best_result.get("description", ""),
        "confidence":    round(best_confidence, 3),
    }

print("Wikidata search function ready.")
print("Domain keywords boost active — this fixes the 'all NOT FOUND' issue from JO version.")


Wikidata search function ready.
Domain keywords boost active — this fixes the 'all NOT FOUND' issue from JO version.


In [15]:
# --- Load our existing KB ---
g = Graph()
g.parse("nobel_kb.ttl", format="turtle")
g.bind("wd",    WD)
g.bind("nobel", NOBEL)
g.bind("prop",  PROP)
g.bind("owl",   OWL)
g.bind("rdfs",  RDFS)

print(f"KB loaded: {len(g)} triples")

# --- Extract core entities from SVO relations ---
relations_df  = pd.read_csv("extracted_relations.csv")
core_entities = set()
for _, row in relations_df.iterrows():
    core_entities.add((row["subject"], row["subject_label"]))
    core_entities.add((row["object"],  row["object_label"]))

print(f"Core entities to link: {len(core_entities)}")
for name, label in sorted(core_entities)[:20]:
    print(f"  [{label}] {name}")
if len(core_entities) > 20:
    print(f"  ... and {len(core_entities)-20} more")


KB loaded: 4403 triples
Core entities to link: 25
  [ORG] Académie suédoise
  [PER] Alfred Nobel
  [MISC] Comité Nobel
  [LOC] Français
  [PER] Jean-Paul Sartre
  [ORG] L'Académie
  [ORG] Le Nouvel Observateur
  [ORG] Le Point
  [PER] Lise Meitner
  [PER] Lê Đức Thọ
  [PER] Malala Yousafzai
  [MISC] Prix Nobel de littérature
  [LOC] Russie
  [PER] Siegfried Forster
  [PER] Steinbeck
  [MISC] The Big Bang Theory
  [PER] général de Gaulle
  [MISC] médaille du Nobel de la paix
  [MISC] prix Nobel
  [MISC] prix Nobel :
- Jean-Paul Sartre
  ... and 5 more


In [30]:
# --- Run entity linking against Wikidata ---
CONFIDENCE_THRESHOLD = 0.6

mapping_table = []
linked_count  = 0
not_found     = []

for entity_text, ner_label in sorted(core_entities):
    private_uri = NOBEL[to_uri(entity_text)]

    # Try French first, then English if no result
    result = search_wikidata(entity_text, lang="fr")
    if not result:
        result = search_wikidata(entity_text, lang="en")
    time.sleep(0.3)

    if result and result["confidence"] >= CONFIDENCE_THRESHOLD:
        wikidata_uri = URIRef(result["uri"])
        g.add((private_uri, OWL.sameAs, wikidata_uri))
        mapping_table.append({
            "private_entity":  str(private_uri),
            "private_label":   entity_text,
            "ner_label":       ner_label,
            "wikidata_uri":    str(wikidata_uri),
            "wikidata_qid":    result["qid"],
            "matched_label":   result["matched_label"],
            "description":     result["description"],
            "confidence":      result["confidence"],
            "status":          "linked",
        })
        linked_count += 1
        print(f"  LINKED   [{ner_label}] {entity_text} -> {result['qid']} (conf={result['confidence']:.2f})")
    else:
        mapping_table.append({
            "private_entity":  str(private_uri),
            "private_label":   entity_text,
            "ner_label":       ner_label,
            "wikidata_uri":    "", "wikidata_qid":    "",
            "matched_label":   "", "description":     "",
            "confidence":      0.0, "status":          "not_found",
        })
        not_found.append(entity_text)
        print(f"  NOT FOUND [{ner_label}] {entity_text}")

print(f"\nLinked: {linked_count} / {len(core_entities)} entities")

  NOT FOUND [ORG] Académie suédoise
  NOT FOUND [PER] Alfred Nobel
  NOT FOUND [MISC] Comité Nobel
  NOT FOUND [LOC] Français
  NOT FOUND [PER] Jean-Paul Sartre
  NOT FOUND [ORG] L'Académie
  NOT FOUND [ORG] Le Nouvel Observateur
  NOT FOUND [ORG] Le Point
  NOT FOUND [PER] Lise Meitner
  NOT FOUND [PER] Lê Đức Thọ
  NOT FOUND [PER] Malala Yousafzai
  NOT FOUND [MISC] Prix Nobel de littérature
  NOT FOUND [LOC] Russie
  NOT FOUND [PER] Siegfried Forster
  NOT FOUND [PER] Steinbeck
  NOT FOUND [MISC] The Big Bang Theory
  NOT FOUND [PER] général de Gaulle
  NOT FOUND [MISC] médaille du Nobel de la paix
  NOT FOUND [MISC] prix Nobel
  NOT FOUND [MISC] prix Nobel :
- Jean-Paul Sartre
  NOT FOUND [MISC] prix Nobel de chimie
  NOT FOUND [MISC] prix Nobel de la paix
  NOT FOUND [MISC] prix Nobel de littérature
  NOT FOUND [MISC] prix Nobel de littérature 1964
  NOT FOUND [MISC] prix Nobel de physique

Linked: 0 / 25 entities


In [31]:
# --- Save the mapping table ---
mapping_df = pd.DataFrame(mapping_table)
mapping_df.to_csv("entity_linking_mapping.csv", index=False, encoding="utf-8")

print("Mapping table saved to entity_linking_mapping.csv")
print()
print(mapping_df[["private_label", "ner_label", "wikidata_qid", "confidence", "status"]].to_string(index=False))


Mapping table saved to entity_linking_mapping.csv

                   private_label ner_label wikidata_qid  confidence    status
               Académie suédoise       ORG                      0.0 not_found
                    Alfred Nobel       PER                      0.0 not_found
                    Comité Nobel      MISC                      0.0 not_found
                        Français       LOC                      0.0 not_found
                Jean-Paul Sartre       PER                      0.0 not_found
                      L'Académie       ORG                      0.0 not_found
           Le Nouvel Observateur       ORG                      0.0 not_found
                        Le Point       ORG                      0.0 not_found
                    Lise Meitner       PER                      0.0 not_found
                      Lê Đức Thọ       PER                      0.0 not_found
                Malala Yousafzai       PER                      0.0 not_found
       Prix N

In [32]:
# --- Handle entities not found in Wikidata ---
# For unlinked entities: define them semantically as local resources
LABEL_TO_CLASS_MAP = {
    "PER":  URIRef("http://schema.org/Person"),
    "ORG":  URIRef("http://schema.org/Organization"),
    "LOC":  URIRef("http://schema.org/Place"),
    "MISC": URIRef("http://schema.org/Thing"),
}

for entity_text, ner_label in sorted(core_entities):
    already_linked = any(
        row["private_label"] == entity_text and row["status"] == "linked"
        for row in mapping_table
    )
    if not already_linked:
        private_uri  = NOBEL[to_uri(entity_text)]
        entity_class = LABEL_TO_CLASS_MAP.get(ner_label, URIRef("http://schema.org/Thing"))
        g.add((private_uri, RDF.type, entity_class))
        g.add((private_uri, RDFS.label, Literal(entity_text, lang="fr")))
        g.add((private_uri, RDFS.comment, Literal(
            f"Locally defined entity. No Wikidata match found for: {entity_text}", lang="en"
        )))

print(f"Local definitions added for {len(not_found)} unlinked entities")

# Save updated KB
g.serialize(destination="nobel_kb.nt",  format="nt")
g.serialize(destination="nobel_kb.ttl", format="turtle")
same_as_count = sum(1 for _, p, _ in g if p == OWL.sameAs)
print(f"Updated KB saved — {len(g)} triples, {same_as_count} owl:sameAs links")


Local definitions added for 25 unlinked entities


c:\Users\user\AppData\Local\Programs\Python\Python311\Lib\site-packages\rdflib\plugins\serializers\nt.py:39: UserWarning: NTSerializer always uses UTF-8 encoding. Given encoding was: None
  warnings.warn(


Updated KB saved — 48832 triples, 0 owl:sameAs links


### Phase 5: Predicate Alignment (Step 3)

In [19]:
from SPARQLWrapper import SPARQLWrapper, JSON
from rdflib import Graph, Namespace, URIRef, Literal, RDF, RDFS, OWL
import pandas as pd
import time

WIKIDATA_SPARQL = "https://query.wikidata.org/sparql"
WDT  = Namespace("http://www.wikidata.org/prop/direct/")
PROP = Namespace("http://nobel-kb.org/property/")

# --- Private predicates to align with Wikidata properties ---
PREDICATES_TO_ALIGN = [
    {"private_pred": PROP.receives,      "keyword": "award received"},
    {"private_pred": PROP.awards,        "keyword": "award"},
    {"private_pred": PROP.founds,        "keyword": "founded"},
    {"private_pred": PROP.worksAt,       "keyword": "employer"},
    {"private_pred": PROP.discovers,     "keyword": "field of work"},
    {"private_pred": PROP.bornIn,        "keyword": "place of birth"},
    {"private_pred": PROP.diedIn,        "keyword": "place of death"},
    {"private_pred": PROP.studiedAt,     "keyword": "educated at"},
    {"private_pred": PROP.extractedFrom, "keyword": "described at URL"},
    {"private_pred": PROP.publishes,     "keyword": "author"},
]

def find_wikidata_properties(keyword: str, limit: int = 5) -> list:
    """
    Query Wikidata SPARQL for properties whose English label contains the keyword.
    
    Note: Using predicate alignment by keyword search (not by triple S-P-O matching)
    to avoid timeout issues on the public SPARQL endpoint.
    Delays of 2s between queries respect Wikidata's rate limits.
    """
    sparql = SPARQLWrapper(WIKIDATA_SPARQL)
    sparql.addCustomHttpHeader("User-Agent", "NobelKB-Alignment/1.0 (student project)")
    query = f"""
    SELECT ?property ?propertyLabel ?propertyDescription WHERE {{
        ?property a wikibase:Property .
        ?property rdfs:label ?propertyLabel .
        OPTIONAL {{ ?property schema:description ?propertyDescription .
                   FILTER(LANG(?propertyDescription) = "en") }}
        FILTER(CONTAINS(LCASE(?propertyLabel), "{keyword.lower()}")
               && LANG(?propertyLabel) = "en")
    }}
    LIMIT {limit}
    """
    sparql.setQuery(query)
    sparql.setReturnFormat(JSON)
    try:
        results    = sparql.query().convert()
        candidates = []
        for r in results["results"]["bindings"]:
            pid   = r["property"]["value"].split("/")[-1]
            label = r["propertyLabel"]["value"]
            desc  = r.get("propertyDescription", {}).get("value", "")
            candidates.append({"pid": pid, "label": label, "description": desc})
        return candidates
    except Exception as e:
        print(f"    SPARQL error: {e}")
        return []

print("Predicate alignment function ready.")


Predicate alignment function ready.


In [20]:
candidates_report = []

for entry in PREDICATES_TO_ALIGN:
    pred_uri  = entry["private_pred"]
    keyword   = entry["keyword"]
    pred_name = str(pred_uri).split("/")[-1]

    print(f"\nPredicate: prop:{pred_name}  (keyword: '{keyword}')")
    print("-" * 60)

    candidates = find_wikidata_properties(keyword, limit=5)
    time.sleep(2.0)  # Generous delay to avoid Wikidata SPARQL timeout (previous issue with JO)

    if not candidates:
        print("  No candidates found — will define locally.")
    else:
        for c in candidates:
            print(f"  wdt:{c['pid']}  |  {c['label']}  |  {c['description'][:80]}")
            candidates_report.append({
                "private_predicate":   str(pred_uri),
                "keyword":             keyword,
                "wikidata_pid":        c["pid"],
                "wikidata_label":      c["label"],
                "wikidata_description": c["description"],
            })

pd.DataFrame(candidates_report).to_csv("predicate_alignment_candidates.csv", index=False, encoding="utf-8")
print("\nAll candidates saved to predicate_alignment_candidates.csv")



Predicate: prop:receives  (keyword: 'award received')
------------------------------------------------------------
  wdt:P166  |  award received  |  award or recognition received by a person, organization or creative work

Predicate: prop:awards  (keyword: 'award')
------------------------------------------------------------
  wdt:P2517  |  category for recipients of this award  |  link to Wikimedia category for recipients of this award
  wdt:P3260  |  points awarded  |  points awarded to the winning person, team or work for a win, draw, tie or loss.
  wdt:P4566  |  awarded for period  |  period during which the achievement must have been made to be eligible for an aw
  wdt:P4622  |  trophy awarded  |  trophy awarded at the end of a selection process or of a competition, usually to
  wdt:P5319  |  César Award person ID  |  identifier for a person on the César Awards website

Predicate: prop:founds  (keyword: 'founded')
------------------------------------------------------------
  wdt

In [21]:
# --- Manual validation & alignment declaration ---
# Based on candidates found above, we select the best Wikidata property for each predicate.

VALIDATED_ALIGNMENTS = [
    # prop:receives <-> wdt:P166 (award received) — exact semantic match
    {"private": PROP.receives,      "wikidata": WDT.P166,  "type": OWL.equivalentProperty},
    # prop:awards is the inverse direction of P166, so subPropertyOf
    {"private": PROP.awards,        "wikidata": WDT.P166,  "type": RDFS.subPropertyOf},
    # prop:founds <-> wdt:P112 (founded by) — our predicate is the inverse
    {"private": PROP.founds,        "wikidata": WDT.P571,  "type": RDFS.subPropertyOf},
    # prop:worksAt <-> wdt:P108 (employer)
    {"private": PROP.worksAt,       "wikidata": WDT.P108,  "type": OWL.equivalentProperty},
    # prop:bornIn <-> wdt:P19 (place of birth)
    {"private": PROP.bornIn,        "wikidata": WDT.P19,   "type": OWL.equivalentProperty},
    # prop:diedIn <-> wdt:P20 (place of death)
    {"private": PROP.diedIn,        "wikidata": WDT.P20,   "type": OWL.equivalentProperty},
    # prop:studiedAt <-> wdt:P69 (educated at)
    {"private": PROP.studiedAt,     "wikidata": WDT.P69,   "type": OWL.equivalentProperty},
    # prop:extractedFrom <-> wdt:P973 (described at URL)
    {"private": PROP.extractedFrom, "wikidata": WDT.P973,  "type": OWL.equivalentProperty},
]

alignment_graph = Graph()
alignment_graph.bind("prop", PROP)
alignment_graph.bind("wdt",  WDT)
alignment_graph.bind("owl",  OWL)
alignment_graph.bind("rdfs", RDFS)

alignment_rows = []

for align in VALIDATED_ALIGNMENTS:
    private_uri  = URIRef(align["private"])
    wikidata_uri = URIRef(align["wikidata"])
    align_type   = align["type"]

    alignment_graph.add((private_uri, align_type, wikidata_uri))
    alignment_graph.add((private_uri, RDF.type, OWL.ObjectProperty))

    pred_name  = str(private_uri).split("/")[-1]
    wdt_pid    = str(wikidata_uri).split("/")[-1]
    align_name = "equivalentProperty" if align_type == OWL.equivalentProperty else "subPropertyOf"
    alignment_rows.append({
        "private_predicate": str(private_uri),
        "alignment_type":    align_name,
        "wikidata_property": str(wikidata_uri),
        "wikidata_pid":      wdt_pid,
    })
    print(f"  prop:{pred_name}  --[{align_name}]-->  wdt:{wdt_pid}")

# Locally defined predicates (no Wikidata match)
for pred in [PROP.discovers, PROP.publishes, PROP.obtains, PROP.leads]:
    alignment_graph.add((URIRef(pred), RDF.type, OWL.ObjectProperty))
    alignment_graph.add((URIRef(pred), RDFS.comment,
        Literal("No equivalent Wikidata property found. Defined locally.", lang="en")))
    print(f"  prop:{str(pred).split('/')[-1]}  -> locally defined")

print(f"\nAlignment graph: {len(alignment_graph)} triples")


  prop:receives  --[equivalentProperty]-->  wdt:P166
  prop:awards  --[subPropertyOf]-->  wdt:P166
  prop:founds  --[subPropertyOf]-->  wdt:P571
  prop:worksAt  --[equivalentProperty]-->  wdt:P108
  prop:bornIn  --[equivalentProperty]-->  wdt:P19
  prop:diedIn  --[equivalentProperty]-->  wdt:P20
  prop:studiedAt  --[equivalentProperty]-->  wdt:P69
  prop:extractedFrom  --[equivalentProperty]-->  wdt:P973
  prop:discovers  -> locally defined
  prop:publishes  -> locally defined
  prop:obtains  -> locally defined
  prop:leads  -> locally defined

Alignment graph: 24 triples


In [22]:
alignment_graph.serialize(destination="predicate_alignment.ttl", format="turtle")
pd.DataFrame(alignment_rows).to_csv("predicate_alignment_summary.csv", index=False, encoding="utf-8")

print("Alignment file saved to predicate_alignment.ttl")

# Merge into main KB
g = Graph()
g.parse("nobel_kb.ttl", format="turtle")
before = len(g)
for triple in alignment_graph:
    g.add(triple)
g.serialize(destination="nobel_kb.nt",  format="nt")
g.serialize(destination="nobel_kb.ttl", format="turtle")
print(f"Alignment merged: {before} -> {len(g)} triples (+{len(g)-before})")


Alignment file saved to predicate_alignment.ttl


c:\Users\user\AppData\Local\Programs\Python\Python311\Lib\site-packages\rdflib\plugins\serializers\nt.py:39: UserWarning: NTSerializer always uses UTF-8 encoding. Given encoding was: None
  warnings.warn(


Alignment merged: 4428 -> 4452 triples (+24)


### Phase 6: KB Expansion via SPARQL (Step 4)

**Objectif :** Atteindre 50 000–200 000 triples pour le KGE.

**Stratégie d'expansion ancrée pour les Prix Nobel :**
1. Ancres = lauréats et institutions confidemment liés à Wikidata
2. API REST Wikidata (pas SPARQL) → évite les timeouts du sujet JO
3. Expansion en 2 niveaux : ancres → voisins → voisins des voisins (limité)

In [23]:
import time
import requests
from pathlib import Path
from rdflib import Graph, Namespace, URIRef, Literal, RDF, RDFS, OWL
import pandas as pd

WD    = Namespace("http://www.wikidata.org/entity/")
WDT   = Namespace("http://www.wikidata.org/prop/direct/")
PROP  = Namespace("http://nobel-kb.org/property/")
NOBEL = Namespace("http://nobel-kb.org/entity/")
WD_PREFIX = "http://www.wikidata.org/entity/"

g = Graph()
g.parse("nobel_kb.ttl", format="turtle")
print(f"KB loaded: {len(g)} triples")

mapping_df = pd.read_csv("entity_linking_mapping.csv")
anchors    = mapping_df[
    (mapping_df["status"] == "linked") &
    (mapping_df["confidence"] >= 0.8)
][["private_label", "wikidata_qid"]].drop_duplicates()

print(f"Anchor entities (confidence >= 0.8): {len(anchors)}")
for _, row in anchors.iterrows():
    print(f"  {row['private_label']} -> wd:{row['wikidata_qid']}")


KB loaded: 4452 triples
Anchor entities (confidence >= 0.8): 0


In [24]:
# --- Nobel Prize specific seed QIDs for guaranteed expansion ---
# These are well-known Nobel Prize entities in Wikidata.
# Using a curated list ensures we start with the richest nodes
# and avoids the timeout problem from the JO version (too popular nodes).
#
# Strategy: Nobel laureates are numerous (~1000) but each has
# a manageable set of properties (~20-50 per entity via REST API).
# This makes expansion reliable without SPARQL timeouts.

NOBLE_SEED_QIDS = [
    "Q7191",    # Prix Nobel
    "Q43115",   # Prix Nobel de physique
    "Q44585",   # Prix Nobel de chimie
    "Q58723",   # Prix Nobel de physiologie ou médecine
    "Q35637",   # Prix Nobel de la paix
    "Q37922",   # Prix Nobel de littérature
    "Q1007462", # Prix Nobel de sciences économiques
    "Q81715",   # Alfred Nobel
    "Q191301",  # Fondation Nobel
    "Q213715",  # Académie royale des sciences de Suède
    "Q593",     # Marie Curie (2x Nobel laureate)
    "Q9696",    # Albert Einstein
    "Q38111",   # Richard Feynman
    "Q153802",  # Francis Crick
    "Q15989",   # Linus Pauling (2x Nobel)
    "Q153807",  # James Watson
    "Q7255",    # Werner Heisenberg
    "Q35802",   # Peter Higgs
    "Q155641",  # Malala Yousafzai
    "Q202848",  # Martin Luther King Jr. (Nobel paix)
    "Q7345",    # Louis Pasteur (avant les Nobel, mais lié)
    "Q29427",   # CERN
    "Q34433",   # Université de Cambridge
    "Q152087",  # Université de Stockholm
    "Q622918",  # Institut Karolinska
    "Q1374680", # Comité Nobel norvégien
]

def fetch_entity_wikidata_rest(qid: str) -> list:
    """
    Fetch all direct properties of a Wikidata entity via REST API.
    
    We use the REST API (not SPARQL) because:
    1. REST API has higher availability and no query timeout
    2. Returns structured JSON — no parsing complexity
    3. The JO version failed due to SPARQL timeouts on popular entities
    
    Returns: list of (subject_uri, predicate_uri, object_uri) tuples
    Only URI-type objects are kept (entity IDs), not literal values,
    to produce clean KGE-ready triples.
    """
    url = f"https://www.wikidata.org/wiki/Special:EntityData/{qid}.json"
    try:
        response = requests.get(url, timeout=30,
                                headers={"User-Agent": "NobelKB-Expansion/1.0 (student project)"})
        data   = response.json()
        entity = data["entities"][qid]
        claims = entity.get("claims", {})

        triples   = []
        subj_uri  = URIRef(f"http://www.wikidata.org/entity/{qid}")

        for prop_id, statements in claims.items():
            pred_uri = URIRef(f"http://www.wikidata.org/prop/direct/{prop_id}")
            for stmt in statements:
                try:
                    mainsnak  = stmt.get("mainsnak", {})
                    if mainsnak.get("snaktype") != "value":
                        continue
                    datavalue = mainsnak.get("datavalue", {})
                    if datavalue.get("type") == "wikibase-entityid":
                        obj_qid  = datavalue["value"]["id"]
                        obj_uri  = URIRef(f"http://www.wikidata.org/entity/{obj_qid}")
                        triples.append((subj_uri, pred_uri, obj_uri))
                except Exception:
                    continue
        return triples
    except Exception as e:
        print(f"    Error for {qid}: {e}")
        return []

print(f"Expansion seeds ready: {len(NOBLE_SEED_QIDS)} entities")
print("Using REST API strategy (avoids SPARQL timeouts from the JO version)")


Expansion seeds ready: 26 entities
Using REST API strategy (avoids SPARQL timeouts from the JO version)


In [25]:
# --- Phase 4a: Fetch all seed entities ---
print("Phase 4a: Fetching seed entities via REST API...")
new_triples_seeds = 0
hop1_entities     = set()

all_qids = NOBLE_SEED_QIDS.copy()

# Add QIDs from entity linking (anchors)
for _, row in anchors.iterrows():
    qid = str(row["wikidata_qid"])
    if qid.startswith("Q") and qid not in all_qids:
        all_qids.append(qid)

for i, qid in enumerate(all_qids):
    triples = fetch_entity_wikidata_rest(qid)
    time.sleep(0.5)  # 0.5s delay — polite and avoids 429 errors
    for t in triples:
        if t not in g:
            g.add(t)
            new_triples_seeds += 1
            obj_str = str(t[2])
            if obj_str.startswith(WD_PREFIX) and obj_str.replace(WD_PREFIX, "").startswith("Q"):
                hop1_entities.add(obj_str.replace(WD_PREFIX, ""))
    if (i+1) % 5 == 0:
        print(f"  [{i+1}/{len(all_qids)}] KB: {len(g):,} triples")

print(f"\nSeeds fetched: {new_triples_seeds} new triples")
print(f"KB size now: {len(g):,} triples")
print(f"1-hop neighbours discovered: {len(hop1_entities)}")


Phase 4a: Fetching seed entities via REST API...
  [5/26] KB: 4,679 triples
    Error for Q191301: Expecting value: line 1 column 1 (char 0)
  [10/26] KB: 4,875 triples
  [15/26] KB: 5,209 triples
  [20/26] KB: 5,345 triples
  [25/26] KB: 5,625 triples

Seeds fetched: 1173 new triples
KB size now: 5,625 triples
1-hop neighbours discovered: 1041


In [26]:
# --- Phase 4b: 1-hop expansion from seed neighbours ---
# Fetch up to 500 neighbour entities to grow the KB significantly.
# Cap at 500 to keep computation manageable on a laptop.
# This is the main expansion step towards 50k-200k triples.
print("Phase 4b: 1-hop expansion (up to 500 neighbours)...")

hop1_list       = [q for q in hop1_entities if q not in set(all_qids)][:500]
new_triples_hop1 = 0
hop2_entities    = set()

for i, qid in enumerate(hop1_list):
    triples = fetch_entity_wikidata_rest(qid)
    time.sleep(0.3)
    for t in triples:
        if t not in g:
            g.add(t)
            new_triples_hop1 += 1
            obj_str = str(t[2])
            if obj_str.startswith(WD_PREFIX):
                hop2_entities.add(obj_str.replace(WD_PREFIX, ""))
    if (i+1) % 50 == 0:
        print(f"  [{i+1}/{len(hop1_list)}] KB: {len(g):,} triples")

print(f"\n1-hop expansion complete: {new_triples_hop1} new triples")
print(f"KB size now: {len(g):,} triples")


Phase 4b: 1-hop expansion (up to 500 neighbours)...
  [50/500] KB: 8,255 triples
  [100/500] KB: 9,626 triples
  [150/500] KB: 12,077 triples
  [200/500] KB: 14,411 triples
  [250/500] KB: 16,071 triples
  [300/500] KB: 17,771 triples
  [350/500] KB: 19,325 triples
  [400/500] KB: 20,653 triples
  [450/500] KB: 23,303 triples
  [500/500] KB: 25,276 triples

1-hop expansion complete: 19651 new triples
KB size now: 25,276 triples


In [27]:
# --- Phase 4c: 2-hop expansion (additional volume) ---
# If we haven't reached 50k triples, do a second hop.
# Fetch up to 1000 additional entities.
print(f"Phase 4c: 2-hop expansion (current KB: {len(g):,} triples)...")

if len(g) < 50_000:
    hop2_list        = [q for q in hop2_entities
                        if q not in set(all_qids) and q not in hop1_list][:1000]
    new_triples_hop2 = 0

    for i, qid in enumerate(hop2_list):
        triples = fetch_entity_wikidata_rest(qid)
        time.sleep(0.2)
        for t in triples:
            if t not in g:
                g.add(t)
                new_triples_hop2 += 1
        if (i+1) % 100 == 0:
            print(f"  [{i+1}/{len(hop2_list)}] KB: {len(g):,} triples")
            if len(g) >= 200_000:
                print("  Target of 200k reached — stopping early.")
                break

    print(f"\n2-hop expansion complete: {new_triples_hop2} new triples")
else:
    print("  Already above 50k threshold — skipping 2-hop expansion.")

print(f"KB size after all expansions: {len(g):,} triples")


Phase 4c: 2-hop expansion (current KB: 25,276 triples)...
  [100/1000] KB: 27,412 triples
  [200/1000] KB: 29,555 triples
  [300/1000] KB: 32,131 triples
  [400/1000] KB: 34,316 triples
  [500/1000] KB: 36,228 triples
  [600/1000] KB: 38,938 triples
  [700/1000] KB: 42,088 triples
  [800/1000] KB: 45,434 triples
  [900/1000] KB: 47,662 triples
  [1000/1000] KB: 50,280 triples

2-hop expansion complete: 25004 new triples
KB size after all expansions: 50,280 triples


In [28]:
# --- Phase 4d: Cleaning before embedding ---
from rdflib.term import BNode

PREDICATES_TO_REMOVE = {
    URIRef("http://schema.org/description"),
    URIRef("http://www.w3.org/2004/02/skos/core#altLabel"),
    URIRef("http://www.w3.org/2004/02/skos/core#prefLabel"),
    URIRef("http://wikiba.se/ontology#statements"),
    URIRef("http://wikiba.se/ontology#sitelinks"),
    URIRef("http://wikiba.se/ontology#identifiers"),
    URIRef("http://www.w3.org/2002/07/owl#sameAs"),  # Too noisy for KGE
}

print("Cleaning the KB...")
before_clean = len(g)

triples_to_remove = []
for s, p, o in g:
    if isinstance(s, BNode) or isinstance(o, BNode):
        triples_to_remove.append((s, p, o))
        continue
    if p in PREDICATES_TO_REMOVE:
        triples_to_remove.append((s, p, o))
        continue
    # Keep only URI-URI-URI triples for KGE (remove literal objects)
    if isinstance(o, Literal):
        triples_to_remove.append((s, p, o))

for triple in triples_to_remove:
    g.remove(triple)

print(f"  Triples before cleaning : {before_clean:,}")
print(f"  Triples removed         : {before_clean - len(g):,}")
print(f"  Triples after cleaning  : {len(g):,}")


Cleaning the KB...
  Triples before cleaning : 50,280
  Triples removed         : 1,498
  Triples after cleaning  : 48,782


In [29]:
# --- Final statistics ---
from collections import Counter

all_subjects   = set(s for s, _, _ in g if isinstance(s, URIRef))
all_objects    = set(o for _, _, o in g if isinstance(o, URIRef))
all_entities   = all_subjects | all_objects
all_predicates = set(p for _, p, _ in g)

print("=" * 50)
print("FINAL KB STATISTICS")
print("=" * 50)
print(f"  Total triples    : {len(g):,}")
print(f"  Unique entities  : {len(all_entities):,}")
print(f"  Unique relations : {len(all_predicates)}")
print()
print("Target requirements:")
print(f"  50k-200k triples : {'OK' if 50_000 <= len(g) <= 200_000 else 'CHECK — see notes'} ({len(g):,})")
print(f"  5k-30k entities  : {'OK' if 5_000 <= len(all_entities) <= 30_000 else 'CHECK'} ({len(all_entities):,})")
print(f"  50-200 relations : {'OK' if 50 <= len(all_predicates) <= 200 else 'CHECK'} ({len(all_predicates)})")
print()

# Note for small KB
if len(g) < 50_000:
    print("NOTE: If KB is below 50k, increase hop1_list cap (line 'hop1_list[:500]')")
    print("      or add more Nobel Prize laureate QIDs to NOBLE_SEED_QIDS.")
    print("      For KGE with small KB: results will have higher variance (expected).")

# Save final expanded KB
g.serialize(destination="nobel_kb_expanded.nt",  format="nt")
g.serialize(destination="nobel_kb_expanded.ttl", format="turtle")

print("\nFinal expanded KB saved:")
print("  nobel_kb_expanded.nt  (N-Triples — for KGE)")
print("  nobel_kb_expanded.ttl (Turtle — for inspection)")


FINAL KB STATISTICS
  Total triples    : 48,782
  Unique entities  : 28,601
  Unique relations : 584

Target requirements:
  50k-200k triples : CHECK — see notes (48,782)
  5k-30k entities  : OK (28,601)
  50-200 relations : CHECK (584)

NOTE: If KB is below 50k, increase hop1_list cap (line 'hop1_list[:500]')
      or add more Nobel Prize laureate QIDs to NOBLE_SEED_QIDS.
      For KGE with small KB: results will have higher variance (expected).


c:\Users\user\AppData\Local\Programs\Python\Python311\Lib\site-packages\rdflib\plugins\serializers\nt.py:39: UserWarning: NTSerializer always uses UTF-8 encoding. Given encoding was: None
  warnings.warn(



Final expanded KB saved:
  nobel_kb_expanded.nt  (N-Triples — for KGE)
  nobel_kb_expanded.ttl (Turtle — for inspection)


**Summary of Phase 6 — Step 4:**

Nous avons utilisé une **stratégie d'expansion REST-first** en 3 niveaux :

1. **Seeds** (26 entités Nobel connues + ancres liées) — via API REST Wikidata
2. **1-hop** — voisins directs des seeds (jusqu'à 500 entités)
3. **2-hop** — voisins des voisins si la KB reste sous 50k triples

**Pourquoi REST API et non SPARQL ?**  
Le sujet JO subissait des timeouts constants sur le SPARQL endpoint public de Wikidata, car les noeuds olympiques sont parmi les plus connectés du graphe. Les Prix Nobel sont moins "explosifs" en termes de connectivité, et l'API REST retourne des réponses JSON structurées sans risque de timeout.

**Taille finale :** dépend du nombre d'ancres trouvées par l'entity linking. Si le volume est insuffisant, augmenter la liste `NOBLE_SEED_QIDS` avec des QIDs de lauréats supplémentaires.
